# Example-14: Convert data (return work)

In [1]:
# Import

import epics
import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import pv_make
from harmonica.window import Window
from harmonica.data import Data

import matplotlib.pyplot as plt
from time import sleep

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Run TEST PVs (softIoc -d virtual_tbt.db)

# Set list of pv names and starting indices

bpm = ['STP2']
pv_list = [pv_make(name, 'X', True) for name in bpm]
pv_rise = [0 for _ in range(len(bpm))]

# Set window

w = Window(4, dtype=dtype, device=device)
print(w)

# Generate test data

frequency = 0.123456
t = torch.linspace(0, w.length-1, w.length, dtype=dtype, device=device)
data = [(10.0 + torch.randn(1, dtype=dtype, device=device))*torch.cos(2.0*numpy.pi*frequency*t) for _ in pv_list]
data = torch.stack(data) + 1.0
data.shape

# Set TbT

d = Data.from_data(w, data)
d.pv_list = pv_list
d.pv_rise = pv_rise
d.source = 'epics'
print(d)

# Convert (print result)

print(d.to_tensor())
print(d.to_numpy())
print(d.to_dict())
print(d.to_frame())

# Clean

del w
del d
if device != torch.device('cpu'):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

Window(4, None, None)
Data(1, Window(4, None, None))
tensor([[1.005246866371e+01, 7.462858018501e+00, 1.175629288121e+00, -5.212082855352e+00]],
       dtype=torch.float64)
[[10.05246866  7.46285802  1.17562929 -5.21208286]]
{'H:STP2:DATA:X': array([10.05246866,  7.46285802,  1.17562929, -5.21208286])}
              PV       DATA
0  H:STP2:DATA:X  10.052469
1  H:STP2:DATA:X   7.462858
2  H:STP2:DATA:X   1.175629
3  H:STP2:DATA:X  -5.212083
